In [1]:
# %% [1] Setup & config
# !pip install -U sentence-transformers torch numpy tqdm

import os, re, json, math, random, zipfile, time, csv
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from sentence_transformers import SentenceTransformer

# ---- basic config ----
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ---- paths (only dev_track_b.jsonl must exist) ----
ITEMS_FILE          = Path("dev_track_b.jsonl")          # provided stories
TRAIN_TRIPLES       = Path("train_triples.jsonl")        # synthetic train triples (we'll create)
SYN_DEV_TRIPLES     = Path("synthetic_dev_triples.jsonl")# synthetic dev triples (for scores)

OUT_JSONL           = Path("track_b.jsonl")
OUT_NPY             = Path("track_b.npy")
OUT_ZIP             = Path("codabench_track_b.zip")

MODEL_DIR           = Path("models"); MODEL_DIR.mkdir(exist_ok=True)
HEAD_PATH           = MODEL_DIR / "nn_head.pt"

BASE_MODEL          = "BAAI/bge-large-en-v1.5"   # frozen encoder
EMBED_DIM           = 1024                       # final embedding dim (10..8192 allowed)
HIDDEN_DIM          = 1024


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
DEVICE: cpu


c:\Users\rishe\anaconda3\envs\rimsh\lib\site-packages\bitsandbytes\cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


In [2]:
# %% [2] IO helpers

def read_items_jsonl(path: Path) -> List[str]:
    """Read one story text per line from dev_track_b.jsonl."""
    assert path.exists(), f"Missing items file: {path}"
    items = []
    with path.open("r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line: 
                continue
            obj = json.loads(line)
            txt = obj.get("text") or obj.get("story") or obj.get("content") or ""
            if not isinstance(txt, str) or not txt.strip():
                raise ValueError(f"Expected string text/story/content on line {ln}")
            items.append(txt.strip())
    print(f"[IO] Read {len(items)} stories from {path}")
    return items

def read_triples_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    triples = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                triples.append(json.loads(line))
    print(f"[IO] Read {len(triples)} triples from {path}")
    return triples


In [3]:
# %% [3] Aspect extraction: theme, action, outcome

SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text: str) -> List[str]:
    text = (text or "").strip()
    if not text:
        return []
    return [s.strip() for s in SENT_SPLIT_RE.split(text) if s.strip()]

def pick_theme(text: str, max_sents: int = 3) -> str:
    """Abstract theme = first few sentences (setup, ideas, motives)."""
    s = split_sentences(text)
    return " ".join(s[:max_sents]) if s else text

ACTION_VERB_CUES = {
    "go","goes","went","travel","travels","traveled","reach","reaches","reached",
    "fight","fights","fought","escape","escapes","escaped","search","searches","searched",
    "find","finds","found","discover","discovers","discovered","return","returns","returned",
    "kill","kills","killed","die","dies","died","plan","plans","planned","attack","attacks","attacked",
    "try","tries","tried","start","starts","started","begin","begins","began","decide","decides","decided",
    "save","saves","saved","help","helps","helped","learn","learns","learned","investigate","investigates","investigated",
    "betray","betrays","betrayed","lose","loses","lost","win","wins","won","organize","organizes","organized"
}

def extract_action_chain(text: str, max_items: int = 6) -> str:
    """Course of action = sentences with action-y verbs."""
    picked, sents = [], split_sentences(text)
    for s in sents:
        toks = re.findall(r"[A-Za-z']+", s.lower())
        if any(t in ACTION_VERB_CUES or t.endswith("ed") or t.endswith("ing") for t in toks):
            picked.append(s)
        if len(picked) >= max_items:
            break
    if not picked and sents:
        picked = sents[:min(3, len(sents))]
    return " | ".join(picked)

def pick_outcome(text: str, max_sents: int = 2) -> str:
    """Outcome = final sentences (results / resolution)."""
    s = split_sentences(text)
    return " ".join(s[-max_sents:]) if s else text

def build_aspect_payload(story: str) -> Dict[str, str]:
    return {
        "theme":   pick_theme(story),
        "action":  extract_action_chain(story),
        "outcome": pick_outcome(story)
    }

# quick sanity on first item (optional)
stories_debug = read_items_jsonl(ITEMS_FILE)[:1]
if stories_debug:
    ap = build_aspect_payload(stories_debug[0])
    print("\n[DEBUG] Example aspects:")
    print("THEME  :", ap["theme"])
    print("ACTION :", ap["action"])
    print("OUTCOME:", ap["outcome"])


[IO] Read 479 stories from dev_track_b.jsonl

[DEBUG] Example aspects:
THEME  : The old grandmother Tina arrives in town to attend the wedding of his nephew Alberto with his girlfriend Ileana. Upon arrival she discovers that she has been stolen of a medallion that her late husband had given her. He goes to the police station to file a complaint and get the dear object back, but given the length of the investigation, he decides to carry out the search for the thief himself, combining a great deal of mess.
ACTION : The old grandmother Tina arrives in town to attend the wedding of his nephew Alberto with his girlfriend Ileana. | Upon arrival she discovers that she has been stolen of a medallion that her late husband had given her. | He goes to the police station to file a complaint and get the dear object back, but given the length of the investigation, he decides to carry out the search for the thief himself, combining a great deal of mess. | Eventually, by chance, he finds the thief, wh

In [4]:
# %% [4] Encoder + aspect encoders (BGE-large)

DOC_PREFIX = "Represent the passage for retrieval: "  # BGE-style instruction
encoder = SentenceTransformer(BASE_MODEL)
BASE_DIM = encoder.get_sentence_embedding_dimension()
print(f"[Init] Loaded encoder: {BASE_MODEL} | base_dim={BASE_DIM}")

def encode_aspects(texts: List[str], batch_size: int = 32) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    For each story, build theme/action/outcome strings & encode separately.
    Returns: E_T, E_A, E_O each of shape (N, BASE_DIM), float32, L2-normalized.
    """
    aspects = [build_aspect_payload(t) for t in texts]
    T = [DOC_PREFIX + "[ASPECT=THEME] "   + a["theme"]   for a in aspects]
    A = [DOC_PREFIX + "[ASPECT=ACTION] "  + a["action"]  for a in aspects]
    O = [DOC_PREFIX + "[ASPECT=OUTCOME] " + a["outcome"] for a in aspects]

    E_T = encoder.encode(T, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True).astype("float32")
    E_A = encoder.encode(A, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True).astype("float32")
    E_O = encoder.encode(O, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True).astype("float32")
    return E_T, E_A, E_O


[Init] Loaded encoder: BAAI/bge-large-en-v1.5 | base_dim=1024


In [5]:
# %% [5] Neural head: gated fusion (theme/action/outcome) + MLP projection

class GatedAspectProjector(nn.Module):
    """
    E_T,E_A,E_O (each D) -> softmax gates -> fused -> MLP -> L2-normalized Z (EMBED_DIM).
    """
    def __init__(self, base_dim: int, out_dim: int, hidden: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(base_dim*3, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 3)
        )
        self.proj = nn.Sequential(
            nn.Linear(base_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, E_T, E_A, E_O):
        cat = torch.cat([E_T, E_A, E_O], dim=-1)      # (B, 3D)
        gates = torch.softmax(self.gate(cat), dim=-1) # (B, 3)
        fused = gates[:,0:1]*E_T + gates[:,1:2]*E_A + gates[:,2:3]*E_O
        Z = self.proj(fused)
        Z = F.normalize(Z, p=2, dim=-1)               # cosine-ready
        return Z, gates


In [7]:
# %% [6] Synthetic train/dev triples from dev_track_b.jsonl

def make_synthetic_train_and_dev(
    items_path: Path, 
    train_path: Path, 
    dev_path: Path, 
    dev_ratio: float = 0.2,
    pos_weight=(0.40,0.35,0.25)
):
    texts = read_items_jsonl(items_path)
    assert len(texts) >= 3, "Need at least 3 items to form triples."

    # Aspect-aware fused embeddings for neighbor selection
    E_T, E_A, E_O = encode_aspects(texts, batch_size=32)
    Z = pos_weight[0]*E_T + pos_weight[1]*E_A + pos_weight[2]*E_O
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)

    sims = Z @ Z.T
    np.fill_diagonal(sims, -np.inf)
    rng = np.random.default_rng(SEED)

    triples = []
    for i in range(len(texts)):
        pos_idx = int(np.argmax(sims[i]))           # closest neighbor
        order   = np.argsort(-sims[i])              # descending similarity
        half    = max(1, len(texts)//2)
        neg_pool = order[half:]                    # far half
        neg_idx  = int(rng.choice(neg_pool)) if len(neg_pool) else int(order[-1])
        triples.append({
            "anchor": texts[i],
            "A": texts[pos_idx],
            "B": texts[neg_idx],
            "text_a_is_closer": True
        })

    # shuffle & split train/dev
    idxs = np.arange(len(triples))
    rng.shuffle(idxs)
    n_dev = int(len(triples) * dev_ratio)
    dev_idxs  = set(idxs[:n_dev])
    train_tr, dev_tr = [], []
    for k, t in enumerate(triples):
        (dev_tr if k in dev_idxs else train_tr).append(t)

    with train_path.open("w", encoding="utf-8") as f:
        for t in train_tr:
            f.write(json.dumps(t) + "\n")
    with dev_path.open("w", encoding="utf-8") as f:
        for t in dev_tr:
            f.write(json.dumps(t) + "\n")

    print(f"[Synthetic] Train triples: {len(train_tr)} → {train_path.resolve()}")
    print(f"[Synthetic] Dev triples  : {len(dev_tr)} → {dev_path.resolve()}")

# always (re)create synthetic triples from dev_track_b.jsonl
make_synthetic_train_and_dev(ITEMS_FILE, TRAIN_TRIPLES, SYN_DEV_TRIPLES)


[IO] Read 479 stories from dev_track_b.jsonl


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

[Synthetic] Train triples: 384 → C:\FALL 2025\NLP\Project\train_triples.jsonl
[Synthetic] Dev triples  : 95 → C:\FALL 2025\NLP\Project\synthetic_dev_triples.jsonl


In [8]:
# %% [7] Training dataset, InfoNCE loss, and head training

def triples_to_pairs(triples: List[Dict[str, Any]]) -> List[Tuple[str,str]]:
    pairs = []
    for t in triples:
        def get(d, ks):
            for k in ks:
                v = d.get(k)
                if isinstance(v, str) and v.strip():
                    return v.strip()
            return ""
        anchor = get(t, ["anchor","Anchor","anchor_text","anchorStory"])
        A      = get(t, ["A","a","candA"])
        B      = get(t, ["B","b","candB"])
        label  = t.get("text_a_is_closer")
        if label is None:
            lab = t.get("label")
            if isinstance(lab,str):
                label = (lab.strip().upper()=="A")
        elif isinstance(label,str):
            label = (label.strip().upper()=="A")
        if anchor and A and B and label is not None:
            pos = A if label else B
            pairs.append((anchor, pos))
    return pairs

class PairDataset(torch.utils.data.Dataset):
    def __init__(self, pairs: List[Tuple[str,str]]):
        # unique texts
        self.txts = sorted(set([t for p in pairs for t in p]))
        self.index = {t:i for i,t in enumerate(self.txts)}
        # pre-encode aspects once
        E_T, E_A, E_O = encode_aspects(self.txts, batch_size=32)
        self.E_T = torch.from_numpy(E_T)
        self.E_A = torch.from_numpy(E_A)
        self.E_O = torch.from_numpy(E_O)
        self.pairs = [(self.index[a], self.index[b]) for a,b in pairs]

    def __len__(self): 
        return len(self.pairs)

    def __getitem__(self, i):
        ia, ib = self.pairs[i]
        return (self.E_T[ia], self.E_A[ia], self.E_O[ia],
                self.E_T[ib], self.E_A[ib], self.E_O[ib])

def info_nce(a, p, temperature: float = 0.07):
    logits = (a @ p.t()) / temperature
    labels = torch.arange(a.size(0), device=a.device)
    return F.cross_entropy(logits, labels)

EPOCHS  = 3
BATCH   = 64
LR      = 3e-4
WARMUP  = 0.1

def train_head(pairs: List[Tuple[str,str]], base_dim: int, out_dim: int) -> GatedAspectProjector:
    model = GatedAspectProjector(base_dim, out_dim).to(DEVICE)
    if not pairs:
        print("[Train] No pairs found; returning random head.")
        return model

    ds = PairDataset(pairs)
    dl = torch.utils.data.DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True)
    opt = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = max(1, len(dl) * EPOCHS)
    warmup_steps = int(total_steps * WARMUP)

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step+1)/max(1,warmup_steps)
        progress = (step - warmup_steps)/max(1, total_steps - warmup_steps)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    model.train()
    for epoch in range(EPOCHS):
        running = 0.0
        for batch in tqdm(dl, desc=f"[Train] Epoch {epoch+1}/{EPOCHS}"):
            aT,aA,aO,bT,bA,bO = [x.to(DEVICE) for x in batch]
            za,_ = model(aT,aA,aO)
            zp,_ = model(bT,bA,bO)
            loss = info_nce(za, zp, 0.07) + info_nce(zp, za, 0.07)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step()
            running += loss.item()
        print(f"  mean loss: {running/len(dl):.4f}")

    torch.save(model.state_dict(), HEAD_PATH)
    print(f"[Train] Saved head → {HEAD_PATH.resolve()}")
    return model


In [9]:
# %% [8] Export Track-B embeddings (per-story inference)

def export_with_head(items_path: Path, model: GatedAspectProjector) -> np.ndarray:
    stories = read_items_jsonl(items_path)
    E_T, E_A, E_O = encode_aspects(stories, batch_size=32)
    with torch.no_grad():
        Z, gates = model(
            torch.from_numpy(E_T).to(DEVICE),
            torch.from_numpy(E_A).to(DEVICE),
            torch.from_numpy(E_O).to(DEVICE)
        )
        X = Z.detach().cpu().numpy().astype("float32")
        G = gates.detach().cpu().numpy().astype("float32")

    with OUT_JSONL.open("w", encoding="utf-8") as f:
        for row in X.tolist():
            f.write(json.dumps({"embeddings": row}) + "\n")
    np.save(OUT_NPY, X)
    with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(OUT_JSONL, arcname="track_b.jsonl")
        z.write(OUT_NPY,   arcname="track_b.npy")

    print(f"[Export] {len(stories)} embeddings → {OUT_JSONL}, {OUT_NPY}")
    print(f"[Export] Created zip → {OUT_ZIP}")

    avg_gates = G.mean(axis=0)
    print(f"[Gates] mean(theme,action,outcome) = {avg_gates.round(3).tolist()}")
    return X

def validate_submission(vecs: np.ndarray, items_file: Path):
    ok = True
    d = vecs.shape[1]
    if not (10 <= d <= 8192):
        print(f"[Check] ❌ Dim {d} not in [10,8192]"); ok=False
    else:
        print(f"[Check] ✅ Dim {d} within limits")
    norms = np.linalg.norm(vecs, axis=1)
    max_dev = float(np.max(np.abs(norms-1.0)))
    if max_dev > 1e-3:
        print(f"[Check] ❌ Norm dev too high: {max_dev:.4e}"); ok=False
    else:
        print(f"[Check] ✅ L2-normalized (max|norm-1|={max_dev:.2e})")
    if not np.isfinite(vecs).all():
        print("[Check] ❌ NaN/Inf found"); ok=False
    else:
        print("[Check] ✅ No NaN/Inf")

    if not OUT_JSONL.exists() or not OUT_NPY.exists() or not OUT_ZIP.exists():
        print("[Check] ❌ Missing export file"); ok=False
    else:
        with zipfile.ZipFile(OUT_ZIP, "r") as z:
            names = set(z.namelist())
        expect = {"track_b.jsonl","track_b.npy"}
        if not expect.issubset(names):
            print(f"[Check] ❌ Zip missing {expect - names}"); ok=False
        else:
            print(f"[Check] ✅ Zip contains {expect}")
    print("[Check] DONE" if ok else "[Check] FAILED")
    return ok


In [10]:
# %% [9] Prediction scores on synthetic dev triples

def _get_text(d: dict, keys):
    for k in keys:
        v = d.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def _label_is_A(d: dict):
    if "text_a_is_closer" in d:
        v = d["text_a_is_closer"]
        if isinstance(v,str): return v.strip().upper()=="A"
        if isinstance(v,(bool,int)): return bool(v)
    if "label" in d:
        v = d["label"]
        if isinstance(v,str): return v.strip().upper()=="A"
        if isinstance(v,(bool,int)): return bool(v)
    return None

def evaluate_and_save(triples_path: Path, model: GatedAspectProjector, csv_out: Path) -> Optional[float]:
    triples = read_triples_jsonl(triples_path)
    if not triples:
        print("[Eval] No triples; skipping.")
        return None

    # collect unique texts
    uniq, seen = [], set()
    for t in triples:
        a = _get_text(t, ["anchor","Anchor","anchor_text","anchorStory"])
        A = _get_text(t, ["A","a","candA"])
        B = _get_text(t, ["B","b","candB"])
        for s in (a,A,B):
            if s and s not in seen:
                seen.add(s); uniq.append(s)

    # embed all with encoder + head once
    E_T, E_A, E_O = encode_aspects(uniq, batch_size=32)
    with torch.no_grad():
        Z,_ = model(torch.from_numpy(E_T).to(DEVICE),
                    torch.from_numpy(E_A).to(DEVICE),
                    torch.from_numpy(E_O).to(DEVICE))
        V = Z.cpu().numpy().astype("float32")
    idx = {txt:i for i,txt in enumerate(uniq)}

    rows = []
    rights, total = 0, 0
    margins = []

    for i, t in enumerate(tqdm(triples, desc="[Eval] scoring")):
        a = _get_text(t, ["anchor","Anchor","anchor_text","anchorStory"])
        A = _get_text(t, ["A","a","candA"])
        B = _get_text(t, ["B","b","candB"])
        lab = _label_is_A(t)
        if not (a and A and B):
            continue
        va, vA, vB = V[idx[a]], V[idx[A]], V[idx[B]]
        sA, sB = float(np.dot(va,vA)), float(np.dot(va,vB))
        pred_is_A = sA >= sB
        gold = None
        if lab is not None:
            gold = "A" if lab else "B"
            total += 1
            rights += int(pred_is_A == lab)
            margins.append(abs(sA - sB))
        rows.append({
            "triple_idx": i,
            "sA": sA,
            "sB": sB,
            "margin": abs(sA - sB),
            "pred": "A" if pred_is_A else "B",
            "gold": gold
        })

    # write CSV
    with csv_out.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"[Eval] Saved scores → {csv_out.resolve()}")

    if total > 0:
        acc = rights / total
        print(f"[Eval] Accuracy on {triples_path.name}: {acc:.4f} (rights={rights}, total={total})")
        if margins:
            print(f"[Eval] Mean confidence margin: {np.mean(margins):.4f}")
        return acc
    else:
        print("[Eval] No gold labels found (should not happen for synthetic triples).")
        return None


In [ ]:


start = time.time()
print("[Run] Starting…")

assert ITEMS_FILE.exists(), f"Items file not found: {ITEMS_FILE.resolve()}"

# 1) load synthetic train/dev triples we created earlier
train_triples = read_triples_jsonl(TRAIN_TRIPLES)
dev_triples   = read_triples_jsonl(SYN_DEV_TRIPLES)

pairs = triples_to_pairs(train_triples)
print(f"[Run] Training pairs: {len(pairs)}")

# 2) train neural head on synthetic train triples
head = train_head(pairs, base_dim=BASE_DIM, out_dim=EMBED_DIM).to(DEVICE)
head.eval()

# 3) export Track-B embeddings for dev_track_b.jsonl
X = export_with_head(ITEMS_FILE, head)
print(f"[Run] Embedding shape: {X.shape} | dtype={X.dtype}")

# 4) validate submission (dim, norms, files)
_ = validate_submission(X, ITEMS_FILE)

# 5) prediction scores + proxy accuracy on synthetic dev triples
pred_csv = Path("synthetic_dev_predictions.csv")
_ = evaluate_and_save(SYN_DEV_TRIPLES, head, pred_csv)

print(f"[Run] Completed in {time.time()-start:.1f}s")


[Run] Starting…
[IO] Read 384 triples from train_triples.jsonl
[IO] Read 95 triples from synthetic_dev_triples.jsonl
[Run] Training pairs: 384


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

[Train] Epoch 1/3:   0%|          | 0/6 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

[Eval] scoring:   0%|          | 0/95 [00:00<?, ?it/s]

[Eval] Saved scores → C:\FALL 2025\NLP\Project\synthetic_dev_predictions.csv
[Eval] Accuracy on synthetic_dev_triples.jsonl: 0.9789 (rights=93, total=95)
[Eval] Mean confidence margin: 0.4545
[Run] Completed in 503.3s
